# Structural Code Summarization — Analysis

Statistical analysis and visualizations for the paper:
- **RQ1**: Compression metrics
- **RQ2**: Task-based evaluation
- **RQ3**: Cost-effectiveness analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 150

## RQ1: Compression Metrics

In [ ]:
# Load RQ1 data
df = pd.read_csv("results/rq1_metrics.csv")
print(f"Total files: {len(df)}")
print(f"Projects: {df['project'].nunique()}")
df.head()

In [ ]:
# Overall compression statistics
metrics = ["compression_ratio", "token_reduction_rate", "information_density"]
summary_stats = df[metrics].describe().T
summary_stats["median"] = df[metrics].median()
print("=== Overall Compression Statistics ===")
print(summary_stats[["mean", "median", "std", "min", "max"]].round(4))

In [ ]:
# Per-project statistics
per_project = df.groupby("project").agg(
    files=("file_name", "count"),
    total_orig_tokens=("original_tokens", "sum"),
    total_summ_tokens=("summary_tokens", "sum"),
    mean_cr=("compression_ratio", "mean"),
    median_cr=("compression_ratio", "median"),
    mean_trr=("token_reduction_rate", "mean"),
    mean_id=("information_density", "mean"),
).round(4)
per_project["overall_cr"] = (per_project["total_summ_tokens"] / per_project["total_orig_tokens"]).round(4)
print("=== Per-Project Statistics ===")
per_project

In [ ]:
# Box plot: Compression Ratio by Project Size Category
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, title in zip(
    axes,
    ["compression_ratio", "token_reduction_rate", "information_density"],
    ["Compression Ratio (lower=better)", "Token Reduction Rate (higher=better)", "Information Density"],
):
    sns.boxplot(data=df, x="project_size_category", y=metric, ax=ax, order=["small", "medium", "large"])
    ax.set_title(title)
    ax.set_xlabel("Project Size")

plt.tight_layout()
plt.savefig("results/rq1_boxplots.png", bbox_inches="tight")
plt.show()

In [ ]:
# Scatter: LOC vs Compression Ratio
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    df["original_lines"],
    df["compression_ratio"],
    c=df["language"].map({"kotlin": "#7F52FF", "java": "#F89820"}),
    alpha=0.5,
    s=20,
)
ax.set_xlabel("Original Lines of Code")
ax.set_ylabel("Compression Ratio")
ax.set_title("File Size vs Compression Ratio")
ax.legend(
    handles=[
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#7F52FF", label="Kotlin"),
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#F89820", label="Java"),
    ]
)
plt.savefig("results/rq1_scatter_loc_cr.png", bbox_inches="tight")
plt.show()

In [ ]:
# Stratify by language
print("=== By Language ===")
lang_stats = df.groupby("language")[metrics].agg(["mean", "median", "std"]).round(4)
print(lang_stats)

## RQ2: Task-Based Evaluation

In [ ]:
import json

# Load RQ2 data
with open("results/rq2_responses.json") as f:
    rq2_raw = json.load(f)

rq2 = pd.DataFrame(rq2_raw)
print(f"Total responses: {len(rq2)}")
print(f"Conditions: {rq2['condition'].unique()}")
rq2.head()

In [ ]:
# Load evaluation scores (manually scored)
# Expected format: CSV with columns: task_id, condition, accuracy_score, completeness_score
try:
    scores = pd.read_csv("results/rq2_scores.csv")
    rq2_scored = rq2.merge(scores, on=["task_id", "condition"])
    print(f"Scored responses: {len(rq2_scored)}")
except FileNotFoundError:
    print("rq2_scores.csv not found — run scoring first")
    rq2_scored = None

In [ ]:
# Token usage comparison across conditions
if "input_tokens" in rq2.columns:
    token_comparison = rq2.groupby("condition").agg(
        mean_input_tokens=("input_tokens", "mean"),
        total_input_tokens=("input_tokens", "sum"),
        mean_output_tokens=("output_tokens", "mean"),
        mean_latency=("latency_seconds", "mean"),
    ).round(2)
    print("=== Token Usage by Condition ===")
    print(token_comparison)

In [ ]:
# Wilcoxon signed-rank test: Full vs Summary quality scores
if rq2_scored is not None:
    full_scores = rq2_scored[rq2_scored["condition"] == "full"].sort_values("task_id")
    summary_scores = rq2_scored[rq2_scored["condition"] == "summary"].sort_values("task_id")

    if len(full_scores) == len(summary_scores):
        # Accuracy
        stat_acc, p_acc = stats.wilcoxon(
            full_scores["accuracy_score"].values,
            summary_scores["accuracy_score"].values,
        )
        print(f"Wilcoxon (accuracy): statistic={stat_acc:.4f}, p={p_acc:.4f}")

        # Completeness
        stat_comp, p_comp = stats.wilcoxon(
            full_scores["completeness_score"].values,
            summary_scores["completeness_score"].values,
        )
        print(f"Wilcoxon (completeness): statistic={stat_comp:.4f}, p={p_comp:.4f}")

        # Cliff's delta (effect size)
        def cliffs_delta(x, y):
            n = len(x) * len(y)
            more = sum(1 for xi in x for yi in y if xi > yi)
            less = sum(1 for xi in x for yi in y if xi < yi)
            return (more - less) / n

        d_acc = cliffs_delta(
            full_scores["accuracy_score"].values,
            summary_scores["accuracy_score"].values,
        )
        d_comp = cliffs_delta(
            full_scores["completeness_score"].values,
            summary_scores["completeness_score"].values,
        )
        print(f"Cliff's delta (accuracy): {d_acc:.4f}")
        print(f"Cliff's delta (completeness): {d_comp:.4f}")
    else:
        print("Mismatched task counts between conditions")

In [ ]:
# Score comparison plot
if rq2_scored is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, score_col, title in zip(
        axes,
        ["accuracy_score", "completeness_score"],
        ["Accuracy Score", "Completeness Score"],
    ):
        sns.boxplot(
            data=rq2_scored,
            x="condition",
            y=score_col,
            order=["full", "summary", "hybrid"],
            ax=ax,
        )
        ax.set_title(title)
        ax.set_ylim(0.5, 5.5)

    plt.tight_layout()
    plt.savefig("results/rq2_scores_boxplot.png", bbox_inches="tight")
    plt.show()

In [ ]:
# Per-category breakdown
if rq2_scored is not None:
    cat_scores = rq2_scored.groupby(["category", "condition"]).agg(
        mean_accuracy=("accuracy_score", "mean"),
        mean_completeness=("completeness_score", "mean"),
    ).round(2)
    print("=== Scores by Category and Condition ===")
    print(cat_scores)

## RQ3: Cost-Effectiveness Analysis

In [ ]:
# Claude pricing (as of 2025)
PRICING = {
    "claude-sonnet-4-20250514": {"input": 3.0 / 1_000_000, "output": 15.0 / 1_000_000},
    "claude-opus-4-20250514": {"input": 15.0 / 1_000_000, "output": 75.0 / 1_000_000},
}

model = "claude-sonnet-4-20250514"  # adjust as needed
price = PRICING[model]

if "input_tokens" in rq2.columns:
    cost_analysis = rq2.groupby("condition").agg(
        total_input=("input_tokens", "sum"),
        total_output=("output_tokens", "sum"),
    )
    cost_analysis["input_cost"] = cost_analysis["total_input"] * price["input"]
    cost_analysis["output_cost"] = cost_analysis["total_output"] * price["output"]
    cost_analysis["total_cost"] = cost_analysis["input_cost"] + cost_analysis["output_cost"]

    print("=== Cost Analysis ===")
    print(cost_analysis.round(4))

    if "full" in cost_analysis.index and "summary" in cost_analysis.index:
        savings = cost_analysis.loc["full", "total_cost"] - cost_analysis.loc["summary", "total_cost"]
        pct = savings / cost_analysis.loc["full", "total_cost"] * 100
        print(f"\nSavings (summary vs full): ${savings:.4f} ({pct:.1f}%)")

In [ ]:
# Projection: daily developer usage
if "input_tokens" in rq2.columns:
    full_data = rq2[rq2["condition"] == "full"]
    summ_data = rq2[rq2["condition"] == "summary"]

    if len(full_data) > 0 and len(summ_data) > 0:
        avg_full_input = full_data["input_tokens"].mean()
        avg_summ_input = summ_data["input_tokens"].mean()

        reads_per_day = [20, 35, 50]  # conservative, typical, heavy
        print("=== Daily Cost Projection (Sonnet) ===")
        for n in reads_per_day:
            daily_full = n * avg_full_input * price["input"]
            daily_summ = n * avg_summ_input * price["input"]
            monthly_savings = (daily_full - daily_summ) * 22  # work days
            print(f"  {n} reads/day: ${daily_full:.2f} (full) vs ${daily_summ:.2f} (summary) — saves ${monthly_savings:.2f}/month")

In [ ]:
# Intra-rater agreement (Cohen's kappa on re-evaluated subset)
try:
    reeval = pd.read_csv("results/rq2_reeval_scores.csv")
    scores_orig = pd.read_csv("results/rq2_scores.csv")

    merged = reeval.merge(scores_orig, on=["task_id", "condition"], suffixes=("_reeval", "_orig"))

    from sklearn.metrics import cohen_kappa_score

    kappa_acc = cohen_kappa_score(
        merged["accuracy_score_orig"], merged["accuracy_score_reeval"], weights="quadratic"
    )
    kappa_comp = cohen_kappa_score(
        merged["completeness_score_orig"], merged["completeness_score_reeval"], weights="quadratic"
    )
    print(f"Intra-rater kappa (accuracy): {kappa_acc:.4f}")
    print(f"Intra-rater kappa (completeness): {kappa_comp:.4f}")
except FileNotFoundError:
    print("Re-evaluation scores not yet available")
except ImportError:
    print("Install scikit-learn for Cohen's kappa: uv pip install scikit-learn")

## Summary Table (for paper)

In [ ]:
# Generate LaTeX table for RQ1
if 'per_project' in dir():
    latex = per_project[["files", "total_orig_tokens", "total_summ_tokens", "overall_cr", "mean_trr"]].to_latex(
        float_format="%.4f",
        caption="Compression metrics per project",
        label="tab:rq1",
    )
    print(latex)